# Recipe Override with SFTTrainer

This notebook demonstrates the **recipe override** feature for fine-tuning with `SFTTrainer`.

The recipe system supports a **3-level precedence** for configuration:

1. **Overrides dict** (highest priority) — passed directly to `SFTTrainer`
2. **Recipe YAML file** — a reusable configuration file
3. **SDK defaults** (lowest priority) — built-in defaults

This means you can define a base recipe in a YAML file and selectively override specific values at runtime.

> **Note:** Actual execution requires valid AWS credentials and an appropriate IAM role. This notebook shows the pattern and API usage.

In [1]:
# Install sagemaker-train from the local staging directory
import sys
import os
sys.path.insert(0, '../../sagemaker-train/src')
sys.path.insert(0, '../../sagemaker-core/src')

# Point botocore at the bundled service model (includes CustomizationTechnique/Peft fields)
os.environ['AWS_DATA_PATH'] = os.path.abspath('../../sagemaker-core/sample')

# Required dependencies
!pip install -q omegaconf pyyaml

/Users/nargokul/.zshenv:1: command not found: 1805912728
zsh:1: command not found: pip


## Step 1: Create a Recipe YAML File

A recipe file defines training hyperparameters and configuration in a reusable YAML format.
Let's write one to disk.

In [2]:
import yaml
import os

# Define the recipe as a Python dict
recipe_config = {
    "training": {
        "learning_rate": 1e-5,
        "num_epochs": 3,
        "batch_size": 8,
        "sequence_length": 2048
    }
}

# Write the recipe to a local YAML file
recipe_path = "my_sft_recipe.yaml"
with open(recipe_path, "w") as f:
    yaml.dump(recipe_config, f, default_flow_style=False)

print(f"Recipe written to: {os.path.abspath(recipe_path)}")
print("\nContents:")
with open(recipe_path) as f:
    print(f.read())

Recipe written to: /Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/my_sft_recipe.yaml

Contents:
training:
  batch_size: 8
  learning_rate: 1.0e-05
  num_epochs: 3
  sequence_length: 2048



## Step 2: Use the Recipe with SFTTrainer and Apply Overrides

We pass both a `recipe` file and an `overrides` dict to `SFTTrainer`.
The overrides will take precedence over recipe file values.

In [3]:
from sagemaker.train import SFTTrainer
from sagemaker.train.common import TrainingType
from sagemaker.train.configs import StoppingCondition

# Create SFTTrainer with recipe file + overrides
sft_trainer = SFTTrainer(
    model="nova-textgeneration-lite-v2",
    training_type=TrainingType.LORA,
    training_dataset="s3://sagemaker-us-west-2-211125564141/recipe-override-test/train.jsonl",
    model_package_group="my-fine-tuned-models",
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
    # Recipe file provides the base configuration
    recipe="my_sft_recipe.yaml",
    # Overrides selectively replace values from the recipe
    overrides={
        "training_config": {
            "learning_rate": 5e-6,   # Override: lower learning rate
            "max_steps": 20          # Override: more steps
        }
    }
)

print("SFTTrainer configured with recipe + overrides.")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/shapes/shapes.py:7865: SyntaxWarning: invalid escape sequence '\|'
  """
/Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/shapes/shapes.py:8683: SyntaxWarning: invalid escape sequence '\*'
  """
/Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/shapes/shapes.py:9039: SyntaxWarning: invalid escape sequence '\*'
  """
/Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-

[06/08/26 12:30:39] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=846208;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=522666;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/nargokul/Library/Application Support/sagemaker/config.yaml


                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=236080;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=402459;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[06/08/26 12:30:40] INFO     SageMaker session not provided. Using default Session.                  ]8;id=505597;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=321963;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=895414;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=960191;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-core/src/sagemaker/core/utils/utils.py#375\375]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=531229;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=449851;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

SFTTrainer configured with recipe + overrides.


## Step 3: Inspect the Resolved Configuration

Use `get_resolved_recipe()` to see the final merged configuration.
This shows exactly what will be used during training, after all precedence rules are applied.

In [4]:
# Inspect the final merged configuration
resolved = sft_trainer.get_resolved_recipe()

print("Resolved recipe (after merging):")
print("=" * 40)
for section, params in resolved.items():
    print(f"\n[{section}]")
    if isinstance(params, dict):
        for key, value in params.items():
            print(f"  {key}: {value}")
    else:
        print(f"  {params}")

Resolved recipe (after merging):

[run]
  name: my-lora-sft-run-vq8ki
  model_type: amazon.nova-2-lite-v1:0:256k
  model_name_or_path: nova-lite-2/prod
  data_s3_path: None
  replicas: 4
  output_s3_path: None
  mlflow_tracking_uri: 
  mlflow_experiment_name: my-lora-sft-experiment
  mlflow_run_name: my-lora-sft-run

[training_config]
  max_steps: 10
  save_steps: 10
  save_top_k: 5
  max_length: 32768
  global_batch_size: 32
  reasoning_enabled: True
  lr_scheduler: {'warmup_steps': 15, 'min_lr': 1e-06}
  optim_config: {'lr': 1e-05, 'weight_decay': 0.0, 'adam_beta1': 0.9, 'adam_beta2': 0.95}
  peft: {'peft_scheme': 'lora', 'lora_tuning': {'alpha': 64, 'lora_plus_lr_ratio': 64.0}}
  model_importance_score: {'fine_tuned_model': 1.0}
  learning_rate: 5e-06
  num_epochs: 5

[training]
  batch_size: 8
  learning_rate: 1e-05
  num_epochs: 3
  sequence_length: 2048


## Step 4: Verify Precedence

Let's explicitly verify the 3-level precedence:

| Parameter | Recipe File | Override | Resolved (Used) | Source |
|-----------|------------|----------|-----------------|--------|
| `learning_rate` | 1e-5 | 5e-6 | **5e-6** | Override wins |
| `max_steps` | 10 (default) | 20 | **20** | Override wins |
| `batch_size` | 8 | (not set) | **8** | Recipe file |
| `sequence_length` | 2048 | (not set) | **2048** | Recipe file |

In [5]:
# Programmatic verification of precedence
resolved = sft_trainer.get_resolved_recipe()

training_config = resolved.get("training_config", resolved.get("training", {}))

print("Resolved training_config:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

# Overrides win over recipe file values
assert training_config["optim_config"]["lr"] == 5e-6, "Override should win"
assert training_config["max_steps"] == 20, "Override should win"

print("\nAll precedence checks passed!")

Resolved training_config:
  max_steps: 10
  save_steps: 10
  save_top_k: 5
  max_length: 32768
  global_batch_size: 32
  reasoning_enabled: True
  lr_scheduler: {'warmup_steps': 15, 'min_lr': 1e-06}
  optim_config: {'lr': 1e-05, 'weight_decay': 0.0, 'adam_beta1': 0.9, 'adam_beta2': 0.95}
  peft: {'peft_scheme': 'lora', 'lora_tuning': {'alpha': 64, 'lora_plus_lr_ratio': 64.0}}
  model_importance_score: {'fine_tuned_model': 1.0}
  learning_rate: 5e-06
  num_epochs: 5

All precedence checks passed!


## Step 4b: Override Non-Spec Keys (Power User)

The full Hub recipe template contains many more parameters than what the UI exposes.
Power users can override **any** key in the full recipe — not just the UI-surfaced subset.

Examples of non-spec keys in the real recipe:
- `max_length` — maximum sequence length for training
- `save_top_k` — number of checkpoint saves to keep
- `optim_config.adam_beta1` — optimizer momentum parameter
- `lr_scheduler.warmup_steps` — learning rate warmup steps

Use `sft_trainer.hyperparameters._full_recipe_template` to inspect all available keys.


In [6]:
# Override non-spec keys that exist in the full recipe template
sft_trainer_power = SFTTrainer(
    model="nova-textgeneration-lite-v2",
    training_type=TrainingType.LORA,
    training_dataset="s3://sagemaker-us-west-2-211125564141/recipe-override-test/train.jsonl",
    model_package_group="my-fine-tuned-models",
    # Override keys that are in the full recipe but NOT in the UI spec
    overrides={
        "training_config": {
            "max_length": 16384,             # Not in UI spec, but in full recipe
            "save_top_k": 3,                 # Not in UI spec, but in full recipe
            "optim_config": {
                "adam_beta1": 0.85            # Nested non-spec key
            },
            "learning_rate": 5e-6,            # In UI spec (also works)
        }
    }
)

# Inspect resolved recipe — non-spec keys are included
resolved_power = sft_trainer_power.get_resolved_recipe()

print("Resolved recipe (power user overrides):")
print("=" * 50)
training_config = resolved_power.get("training_config", {})
for key, value in sorted(training_config.items()):
    if isinstance(value, dict):
        print(f"  {key}:")
        for k2, v2 in value.items():
            print(f"    {k2}: {v2}")
    else:
        print(f"  {key}: {value}")

# Verify non-spec keys are present and overridden
assert training_config["max_length"] == 16384, "Non-spec key override should work"
assert training_config["save_top_k"] == 3, "Non-spec key override should work"
assert training_config["optim_config"]["adam_beta1"] == 0.85, "Nested non-spec override should work"
assert training_config["optim_config"]["adam_beta2"] == 0.95, "Unchanged default preserved"
print("All power user overrides applied successfully!")


[06/08/26 12:30:41] INFO     SageMaker session not provided. Using default Session.                  ]8;id=701999;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=820258;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#61\61]8;;\

Resolved recipe (power user overrides):
  global_batch_size: 32
  learning_rate: 5e-06
  lr_scheduler:
    warmup_steps: 15
    min_lr: 1e-06
  max_length: 16384
  max_steps: 10
  model_importance_score:
    fine_tuned_model: 1.0
  optim_config:
    lr: 1e-05
    weight_decay: 0.0
    adam_beta1: 0.85
    adam_beta2: 0.95
  peft:
    peft_scheme: lora
    lora_tuning: {'alpha': 64, 'lora_plus_lr_ratio': 64.0}
  reasoning_enabled: True
  save_steps: 10
  save_top_k: 3
All power user overrides applied successfully!


## Step 5: The "No Recipe" Path (Hyperparameters Property)

If you prefer not to use recipe files at all, you can still set hyperparameters directly
using the `hyperparameters` property. This path continues to work as before.

In [7]:
# Alternative: No recipe file, just direct hyperparameters
sft_trainer_simple = SFTTrainer(
    model="nova-textgeneration-lite-v2",
    training_type=TrainingType.LORA,
    training_dataset="s3://sagemaker-us-west-2-211125564141/recipe-override-test/train.jsonl",
    model_package_group="my-fine-tuned-models",
)

# Set hyperparameters directly (existing approach, still fully supported)
sft_trainer_simple.hyperparameters.learning_rate = 2e-5

print("SFTTrainer configured without recipe (hyperparameters property).")
print(f"  learning_rate: {sft_trainer_simple.hyperparameters.learning_rate}")

[06/08/26 12:30:42] INFO     SageMaker session not provided. Using default Session.                  ]8;id=561288;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=18015;file:///Users/nargokul/workspace/nova-forge-integration/staging/v3-examples/model-customization-examples/../../sagemaker-train/src/sagemaker/train/defaults.py#61\61]8;;\

SFTTrainer configured without recipe (hyperparameters property).
  learning_rate: 2e-05


## Step 6: Launch Training (Requires AWS Credentials)

When you're ready to run the job, call `.train()`. This submits the training job to SageMaker.

In [8]:
# Launch training
training_job = sft_trainer.train(wait=True)

print(f"Training job completed: {training_job}")

╭────────────────────────────────── Training Job Status ───────────────────────────────────╮
│  TrainingJob Name      nova-textgeneration-lite-v2-sft-20260608123043                    │
│  TrainingJob ARN       arn:aws:sagemaker:us-west-2:211125564141:training-job/nova-textg  │
│                        eneration-lite-v2-sft-20260608123043                              │
│  Links                 ]8;id=53546;https://us-west-2.console.aws.amazon.com/sagemaker/home?region=us-west-2#/jobs/nova-textgeneration-lite-v2-sft-20260608123043\🔗 Training Job (Console)]8;;\                                         │
│                        ]8;id=158196;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3Dnova-textgeneration-lite-v2-sft-20260608123043\🔗 CloudWatch Logs]8;;\                                                │
│                                                                                          │
│  Job Status            Failed                                                            │
│  Secondary Status      Failed                                                            │
│  Elapsed Time          586.5s                                                            │
│  Failure Reason        ClientError: Invalid input error: Number of samples 5 out of      │
│                        bounds between 32 and inf for amazon.nova-2-lite-v1:0:256k, exit  │
│                        code: 0                                                           │
│                                                                                          │
│ Status Transitions                                                                       │
│                                                                                          │
│        Step              Details                               Duration                  │
│  ───────────────────────────────────────────────────────────────────────────             │
│   ✓    Starting          Starting the training job             2.6s                      │
│   ✓    Pending           Preparing the instances for           168.4s                    │
│                          training                                                        │
│   ✓    Downloading       Downloading the training image        338.7s                    │
│   ✓    Training          Training image download completed.    50.8s                     │
│                          Training in progress.                                           │
│   ✓    Uploading         Uploading generated training model    28.0s                     │
│   ✓    Failed            Training job failed                   0.0s                      │
│                                                                                          │
╰──────────────────────────────────────────────────────────────────────────────────────────╯

Training job completed: training_job_name='nova-textgeneration-lite-v2-sft-20260608123043' training_job_arn='arn:aws:sagemaker:us-west-2:211125564141:training-job/nova-textgeneration-lite-v2-sft-20260608123043' tuning_job_arn=Unassigned() labeling_job_arn=Unassigned() auto_ml_job_arn=Unassigned() model_artifacts=Unassigned() training_job_status='Failed' secondary_status='Failed' failure_reason='ClientError: Invalid input error: Number of samples 5 out of bounds between 32 and inf for amazon.nova-2-lite-v1:0:256k, exit code: 0' hyper_parameters={'batch_size': '8', 'fine_tuned_model': '1.0', 'global_batch_size': '32', 'learning_rate': '1e-05', 'learning_rate_ratio': '64.0', 'lora_alpha': '64', 'max_context_length': '8192', 'max_length': '32768', 'max_steps': '10', 'min_lr': '1e-06', 'mlflow_experiment_name': 'my-lora-sft-experiment', 'mlflow_run_name': 'my-lora-sft-run', 'model_name_or_path': 'nova-lite-2/prod', 'model_type': 'amazon.nova-2-lite-v1:0:256k', 'name': 'my-lora-sft-run-vq8ki

## Summary

The recipe override system provides:

- **Reusability** — Define base configs in YAML files, share across team
- **Flexibility** — Override any parameter at runtime without editing the file
- **Full recipe access** — Power users can override **any** key in the full Hub recipe, not just the UI-exposed subset (e.g., `sequence_length`, `warmup_ratio`, `gradient_accumulation_steps`)
- **Transparency** — `get_resolved_recipe()` shows exactly what will be used
- **Backward compatibility** — The `hyperparameters` property still works

**Precedence (highest to lowest):**
1. `overrides` dict
2. Recipe YAML file
3. Full Hub recipe defaults (all parameters)
